# Deep Learning Inference for Mass Regression

This notebook implements a Convolutional Neural Network (ConvNet) for mass regression using parquet data.
Key features:
1.  Modified `ConvNet` architecture with Batch Normalization in the first layer.
2.  Normalized target labels (Z-score normalization).
3.  Training loop that iterates over parquet files.
4.  Learning Rate scheduler that decays by 0.9 after processing each file.

In [30]:
import os
import glob
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, IterableDataset
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Check for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
def _to_fixed_jet_array(x_item, target_channels=4, jet_h=125, jet_w=125):
    """Convert a single X_jet sample to a fixed (C,H,W) float32 array.

    Handles:
      - (H,W) -> adds channel dim
      - (C,H,W) -> crops/pads channels/spatial
      - list/tuple of channels (each (H,W) or flat length H*W)
      - Arrow scalars (via .as_py())

    Never raises on ragged inputs; returns zeros if unparseable.
    """
    if hasattr(x_item, "as_py"):
        x_item = x_item.as_py()

    out = np.zeros((target_channels, jet_h, jet_w), dtype=np.float32)

    # Try direct conversion first (fast when structure is regular).
    x = None
    try:
        x = np.asarray(x_item, dtype=np.float32)
    except (ValueError, TypeError):
        x = None

    if x is not None:
        # If conversion produced an object array, treat as ragged and fall back.
        if x.dtype == object:
            x = None
        else:
            if x.ndim == 2:
                x = x[np.newaxis, ...]
            if x.ndim == 3:
                c = min(x.shape[0], target_channels)
                h = min(x.shape[1], jet_h)
                w = min(x.shape[2], jet_w)
                out[:c, :h, :w] = x[:c, :h, :w]
                return out
            # Some encodings end up as flat vectors; try reshape if plausible.
            if x.ndim == 1 and x.size == jet_h * jet_w:
                out[0, :, :] = x.reshape(jet_h, jet_w)
                return out
            # Otherwise fall through to channel-wise parsing.

    # Channel-wise parsing: treat x_item as sequence of channels.
    if isinstance(x_item, (list, tuple)) and len(x_item) > 0:
        filled_any = False
        for ch_idx in range(min(len(x_item), target_channels)):
            ch = x_item[ch_idx]
            if hasattr(ch, "as_py"):
                ch = ch.as_py()
            try:
                ch_arr = np.asarray(ch, dtype=np.float32)
            except (ValueError, TypeError):
                continue
            if ch_arr.dtype == object:
                continue
            if ch_arr.ndim == 1 and ch_arr.size == jet_h * jet_w:
                ch_arr = ch_arr.reshape(jet_h, jet_w)
            elif ch_arr.ndim > 2:
                # If extra dims exist, try to squeeze or take first slice.
                ch_arr = np.squeeze(ch_arr)
                if ch_arr.ndim > 2:
                    ch_arr = ch_arr.reshape(-1, ch_arr.shape[-1])
            if ch_arr.ndim != 2:
                continue
            h = min(ch_arr.shape[0], jet_h)
            w = min(ch_arr.shape[1], jet_w)
            out[ch_idx, :h, :w] = ch_arr[:h, :w]
            filled_any = True
        if filled_any:
            return out

    # Completely unparseable -> all zeros.
    return out

class ParquetDataset(Dataset):
    def __init__(self, file_path, target_mean=None, target_std=None, target_channels=4, chunk_size=4096, batch_size=512):
        self.file_path = file_path
        self.target_mean = target_mean if target_mean is not None else 0.0
        self.target_std = target_std if target_std is not None else 1.0
        self.target_channels = target_channels
        self.chunk_size = chunk_size
        self.batch_size = batch_size

        pf = pq.ParquetFile(self.file_path)
        num_rows = pf.metadata.num_rows
        if num_rows == 0:
            raise ValueError(f"No rows found in {self.file_path}")

        safe_std = self.target_std if abs(self.target_std) > 1e-12 else 1.0
        jet_h, jet_w = 125, 125
        x_chunks = []
        y_chunks = []
        bad_samples = 0

        for batch in pf.iter_batches(columns=['X_jet', 'm'], batch_size=self.chunk_size):
            x_vals = batch.column(0).to_pylist()
            y_vals = np.asarray(batch.column(1).to_numpy(zero_copy_only=False), dtype=np.float32)
            n = len(x_vals)
            if n == 0:
                continue

            x_chunk = np.zeros((n, self.target_channels, jet_h, jet_w), dtype=np.float32)

            # Fast path per chunk when shapes are consistent.
            x_all = None
            try:
                x_all = np.asarray(x_vals, dtype=np.float32)
            except (ValueError, TypeError):
                x_all = None

            if x_all is not None and x_all.ndim == 4 and x_all.dtype != object:
                in_channels = x_all.shape[1]
                c = min(in_channels, self.target_channels)
                h = min(x_all.shape[2], jet_h)
                w = min(x_all.shape[3], jet_w)
                x_chunk[:, :c, :h, :w] = x_all[:, :c, :h, :w]
            else:
                for i, x_item in enumerate(x_vals):
                    fixed = _to_fixed_jet_array(
                        x_item,
                        target_channels=self.target_channels,
                        jet_h=jet_h,
                        jet_w=jet_w,
                    )
                    # If it is all zeros, treat as malformed (keeps old behavior: zero-fill but warn).
                    if not np.any(fixed):
                        bad_samples += 1
                    x_chunk[i] = fixed

            y_norm = (y_vals - self.target_mean) / safe_std
            x_chunks.append(x_chunk)
            y_chunks.append(y_norm.astype(np.float32, copy=False))

            del x_vals
            del y_vals
            del y_norm
            if x_all is not None:
                del x_all

        if len(x_chunks) == 0:
            raise ValueError(f"No usable rows found in {self.file_path}")

        self.x_data = torch.from_numpy(np.concatenate(x_chunks, axis=0))
        self.y_data = torch.from_numpy(np.concatenate(y_chunks, axis=0)).unsqueeze(1)

        if bad_samples > 0:
            print(f"Warning: {bad_samples} malformed X_jet samples in {os.path.basename(self.file_path)} were zero-filled.")

        del x_chunks
        del y_chunks

    def __len__(self):
        return self.x_data.size(0)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]


def _combine_stats(count_a, mean_a, m2_a, count_b, mean_b, m2_b):
    if count_b == 0:
        return count_a, mean_a, m2_a
    if count_a == 0:
        return count_b, mean_b, m2_b
    delta = mean_b - mean_a
    total = count_a + count_b
    mean = mean_a + delta * count_b / total
    m2 = m2_a + m2_b + delta * delta * count_a * count_b / total
    return total, mean, m2


def compute_global_stats(files, batch_size=200000):
    print("Computing global stats (mean/std) for 'm'...")
    total_count = 0
    global_mean = 0.0
    global_m2 = 0.0

    for f in files:
        pf = pq.ParquetFile(f)
        file_count = 0
        file_mean = 0.0
        file_m2 = 0.0

        for batch in pf.iter_batches(columns=['m'], batch_size=batch_size):
            m_vals = batch.column(0).to_numpy(zero_copy_only=False)
            if len(m_vals) == 0:
                continue
            batch_count = len(m_vals)
            batch_mean = float(np.mean(m_vals))
            batch_m2 = float(np.var(m_vals, ddof=0) * batch_count)
            file_count, file_mean, file_m2 = _combine_stats(
                file_count, file_mean, file_m2, batch_count, batch_mean, batch_m2
            )

        if file_count == 0:
            print(f"File: {os.path.basename(f)}, Count: 0 (skipped)")
            continue
        file_std = np.sqrt(file_m2 / file_count)
        print(f"File: {os.path.basename(f)}, Count: {file_count}, Mean: {file_mean:.2f}")

        total_count, global_mean, global_m2 = _combine_stats(
            total_count, global_mean, global_m2, file_count, file_mean, file_m2
        )

    if total_count == 0:
        raise ValueError("No rows found in training files to compute stats.")
    global_std = np.sqrt(global_m2 / total_count)

    print(f"Global Mean: {global_mean:.4f}, Global Std: {global_std:.4f}")
    return global_mean, global_std

In [ ]:
sample_files = sorted(glob.glob('/kaggle/input/datasets/vishalreddyk/brokendataset/*.parquet'))
if not sample_files:
    print("Sanity check skipped: no parquet files found in brokenfilesdata/.")
else:
    f = sample_files[0]
    print("Sanity check file:", os.path.basename(f))
    pf = pq.ParquetFile(f)
    batch = next(pf.iter_batches(columns=['X_jet', 'm'], batch_size=4))
    x_vals = batch.column(0).to_pylist()
    y_vals = batch.column(1).to_numpy()
    for i, x_item in enumerate(x_vals):
        fixed = _to_fixed_jet_array(x_item, target_channels=4, jet_h=125, jet_w=125)
        print(f"  sample {i}: X {fixed.shape} {fixed.dtype}, nonzero={int(np.count_nonzero(fixed))}, m={float(y_vals[i])}")

In [22]:
# 1. File split and stats from brokenfilesdata
all_files = sorted(glob.glob('brokenfilesdata/*.parquet'))
print(f"Found {len(all_files)} files: {[os.path.basename(f) for f in all_files]}")

if len(all_files) == 0:
    raise ValueError("No parquet files found in brokenfilesdata/.")

train_count = max(1, int(len(all_files) * 0.8))
train_files = all_files[:train_count]
heldout_files = all_files[train_count:]

print(f"Training on {len(train_files)} files (80%): {[os.path.basename(f) for f in train_files]}")
print(f"Held-out files ({len(heldout_files)}): {[os.path.basename(f) for f in heldout_files]}")

# IMPORTANT: stats from TRAIN files only to avoid leakage
global_mean, global_std = compute_global_stats(train_files)

Found 140 files: ['top_gun_opendata_0_part_00000.parquet', 'top_gun_opendata_0_part_00001.parquet', 'top_gun_opendata_0_part_00002.parquet', 'top_gun_opendata_0_part_00003.parquet', 'top_gun_opendata_0_part_00004.parquet', 'top_gun_opendata_0_part_00005.parquet', 'top_gun_opendata_0_part_00006.parquet', 'top_gun_opendata_0_part_00007.parquet', 'top_gun_opendata_0_part_00008.parquet', 'top_gun_opendata_0_part_00009.parquet', 'top_gun_opendata_0_part_00010.parquet', 'top_gun_opendata_0_part_00011.parquet', 'top_gun_opendata_0_part_00012.parquet', 'top_gun_opendata_0_part_00013.parquet', 'top_gun_opendata_0_part_00014.parquet', 'top_gun_opendata_0_part_00015.parquet', 'top_gun_opendata_0_part_00016.parquet', 'top_gun_opendata_0_part_00017.parquet', 'top_gun_opendata_0_part_00018.parquet', 'top_gun_opendata_0_part_00019.parquet', 'top_gun_opendata_1_part_00000.parquet', 'top_gun_opendata_1_part_00001.parquet', 'top_gun_opendata_1_part_00002.parquet', 'top_gun_opendata_1_part_00003.parquet'

In [23]:
# 2. Initialize Model, Optimizer, and Scheduler
# Changed inplanes to 4 based on requirement "use 4 channels at most".
# Our Dataset class now ensures all inputs are exactly 4 channels.
from models import convnet
model = convnet(layers=[2, 2, 2, 2], inplanes=4, expansion=1).to(device)
criterion = nn.MSELoss()
initial_lr = 1e-3

optimizer = optim.Adam(model.parameters(), lr=initial_lr)

# Exponential LR Scheduler: Decay by gamma=0.9 every epoch
gamma = 0.9
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

print("Model Initialized with 4 input channels.")
print(f"Initial LR: {optimizer.param_groups[0]['lr']}")

Model Initialized with 4 input channels.
Initial LR: 0.001


In [24]:
def train_on_file(file_idx, file_path, epochs=10):

    print(f"\n--- Training on File {file_idx+1}: {os.path.basename(file_path)} ---")
    print(f"Current Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    train_dataset = ParquetDataset(
        file_path,
        target_mean=global_mean,
        target_std=global_std,
        target_channels=4,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=512,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    try:
        for epoch in range(epochs):

            model.train()

            running_loss = 0.0
            total_samples = 0

            for batch_idx, (inputs, targets) in enumerate(train_loader):

                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                optimizer.zero_grad()

                outputs = model(inputs)

                loss = criterion(outputs, targets)

                loss.backward()

                optimizer.step()

                running_loss += loss.item() * inputs.size(0)

                total_samples += inputs.size(0)

                if batch_idx % 100 == 0:
                    print(f"Epoch {epoch+1}/{epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}")

            avg_loss = running_loss / max(total_samples, 1)

            print(f"Epoch {epoch+1}/{epochs} Training Loss: {avg_loss:.4f}")

            scheduler.step()

            print(f"New LR: {optimizer.param_groups[0]['lr']:.6f}")

            torch.save(model.state_dict(), f'latest_checkpoint_file_{file_idx+1}.pth')

            print(f"Latest checkpoint saved: latest_checkpoint_file_{file_idx+1}.pth")
    finally:
        # Release file-level data structures after all epochs for this file.
        del train_loader
        del train_dataset
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f"Finished processing file {file_idx+1}")

In [ ]:
for file_idx, file_path in enumerate(train_files):
    train_on_file(file_idx, file_path, epochs=10)


--- Training on File 1: top_gun_opendata_0_part_00000.parquet ---
Current Learning Rate: 0.001000


KeyboardInterrupt: 

In [ ]:
state_dict = torch.load(f'latest_checkpoint_file_{1}.pth', map_location=device)
model.load_state_dict(state_dict)

train_on_file(0, train_files[0]) # File 1 with File 7 validation


--- Training on File 1: top_gun_opendata_0.parquet ---
Current Learning Rate: 0.001000


TypeError: ParquetIterableDataset.__init__() got an unexpected keyword argument 'batch_size'

In [ ]:
state_dict = torch.load(f'latest_checkpoint_file_{1}.pth', map_location=device)
model.load_state_dict(state_dict)

train_on_file(1, train_files[1], ) # File 2 with File 7 validation


--- Training on File 2: top_gun_opendata_1.parquet ---
Current Learning Rate: 0.000066
Epoch 1/4, Batch 0, Loss: 0.5516


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001AED2DD18E0>>
Traceback (most recent call last):
  File "c:\Users\visha\anaconda3\envs\learning_pytorch\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\visha\anaconda3\envs\learning_pytorch\Lib\threading.py", line 1535, in enumerate
    def enumerate():
    
KeyboardInterrupt: 
